<a href="https://colab.research.google.com/github/kangwonlee/nmisp/blob/main/40_linear_algebra_1/38_QR_decomposition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# QR Decomposition<br>QR 분해

그람-슈미트 직교화는 벡터들을 정규직교 집합으로 바꾸는 과정이었다. 그 과정을 행렬 $A$ 의 **열벡터** 들에 적용하면, 부산물로 두 개의 행렬이 떨어진다 — 직교행렬 $Q$ 와 상삼각행렬 $R$. 이것이 **QR 분해** $A = QR$ 이다.<br>
The Gram-Schmidt process turned a set of vectors into an orthonormal set. If we run that same process on the **columns** of a matrix $A$, two matrices fall out as a by-product — an orthogonal matrix $Q$ and an upper-triangular matrix $R$. That is the **QR decomposition** $A = QR$.


In [ ]:
# 행렬과 수치 계산 기능
# Matrix and numerical features
import numpy as np
import numpy.linalg as nl


## 핵심 아이디어<br>The core idea

행렬 $A$ 의 열을 $\mathbf{a}_0, \mathbf{a}_1, \dots$ 라 하자. 각 열에 그람-슈미트를 적용하면 정규직교 벡터 $\mathbf{q}_0, \mathbf{q}_1, \dots$ 를 얻는다. 이 $\mathbf{q}$ 들이 $Q$ 의 열이 된다.<br>
Let the columns of $A$ be $\mathbf{a}_0, \mathbf{a}_1, \dots$. Running Gram-Schmidt on them gives orthonormal vectors $\mathbf{q}_0, \mathbf{q}_1, \dots$, and these become the columns of $Q$.

각 열 $\mathbf{a}_k$ 는 자기보다 앞선 정규직교 기저들의 선형결합으로 정확히 복원된다:<br>
Each column $\mathbf{a}_k$ is recovered exactly as a combination of the orthonormal vectors up to index $k$:

$$
\mathbf{a}_k = \sum_{i=0}^{k} r_{ik}\,\mathbf{q}_i,
\qquad r_{ik} = \mathbf{q}_i \cdot \mathbf{a}_k .
$$

이 계수 $r_{ik}$ 를 모으면 상삼각행렬 $R$ 이다. $k$ 보다 큰 행은 0 이므로 위쪽 삼각형만 채워진다.<br>
Collecting the coefficients $r_{ik}$ gives the upper-triangular matrix $R$: rows below $k$ are zero, so only the upper triangle is filled.


## 예제 행렬<br>An example matrix

깔끔한 정수 결과가 나오는 고전적인 예제를 쓰자.<br>
Let's use a classic example that yields clean integer results.


In [ ]:
matA = np.array([
    [12.0, -51.0,   4.0],
    [ 6.0, 167.0, -68.0],
    [-4.0,  24.0, -41.0],
])
matA


## 그람-슈미트로 $Q$ 와 $R$ 만들기<br>Building $Q$ and $R$ with Gram-Schmidt

각 열에 대해: 앞선 $\mathbf{q}_i$ 방향 성분 $r_{ik}=\mathbf{q}_i\cdot\mathbf{a}_k$ 를 빼고, 남은 벡터의 크기가 $r_{kk}$, 그것을 정규화한 것이 $\mathbf{q}_k$ 이다.<br>
For each column: subtract the components $r_{ik}=\mathbf{q}_i\cdot\mathbf{a}_k$ along the earlier $\mathbf{q}_i$; the length of what remains is $r_{kk}$, and normalizing it gives $\mathbf{q}_k$.


In [ ]:
def gram_schmidt_qr(matA):
    """그람-슈미트로 A = Q R 분해 / QR decomposition of A by Gram-Schmidt."""
    n_row, n_col = matA.shape
    matQ = np.zeros((n_row, n_col))
    matR = np.zeros((n_col, n_col))

    for k in range(n_col):
        # k 번째 열에서 시작 / start from the k-th column
        v = matA[:, k].astype(float).copy()

        for i in range(k):
            # 앞선 q_i 방향 성분의 계수 / coefficient along the earlier q_i
            matR[i, k] = matQ[:, i] @ matA[:, k]
            # 그 성분을 빼낸다 / remove that component
            v = v - matR[i, k] * matQ[:, i]

        # 남은 벡터의 크기가 대각 성분 / the remaining length is the diagonal entry
        matR[k, k] = nl.norm(v)
        # 정규화하여 q_k / normalize to get q_k
        matQ[:, k] = v / matR[k, k]

    return matQ, matR


In [ ]:
matQ, matR = gram_schmidt_qr(matA)


In [ ]:
# 직교행렬 Q / the orthogonal matrix Q
matQ


In [ ]:
# 상삼각행렬 R / the upper-triangular matrix R
matR


## 확인 1 : $A = QR$ 인가?<br>Check 1: does $A = QR$?


In [ ]:
matQ @ matR


In [ ]:
assert np.allclose(matA, matQ @ matR), "A != QR"
print("A = Q R  확인 / verified")


## 확인 2 : $Q$ 의 열은 정규직교인가?<br>Check 2: are the columns of $Q$ orthonormal?

정규직교라면 $Q^{T} Q = I$ (단위행렬) 이다.<br>
If they are orthonormal then $Q^{T} Q = I$, the identity matrix.


In [ ]:
matQ.T @ matQ


In [ ]:
assert np.allclose(matQ.T @ matQ, np.eye(matA.shape[1])), "Q is not orthonormal"
print("Q^T Q = I  확인 / verified")


## 확인 3 : $R$ 은 상삼각인가?<br>Check 3: is $R$ upper-triangular?

대각선 아래 성분이 모두 0 이어야 한다.<br>
Every entry below the diagonal must be zero.


In [ ]:
assert np.allclose(np.tril(matR, k=-1), 0), "R is not upper-triangular"
print("R 의 대각선 아래는 0 / below-diagonal of R is zero")


## NumPy 의 결과와 비교<br>Compare with NumPy

`numpy.linalg.qr` 도 같은 분해를 계산한다. 단, $Q$ 의 각 열과 $R$ 의 각 행의 **부호** 는 구현마다 다를 수 있다 ($\mathbf{q}_k$ 와 $-\mathbf{q}_k$ 모두 정규직교이기 때문). 그래서 크기(절댓값)를 비교한다.<br>
`numpy.linalg.qr` computes the same decomposition. The **sign** of each column of $Q$ (and the matching row of $R$) may differ between implementations — both $\mathbf{q}_k$ and $-\mathbf{q}_k$ are orthonormal — so we compare magnitudes.


In [ ]:
matQ_np, matR_np = nl.qr(matA)
matR_np


In [ ]:
# 부호를 제외하면 R 은 일치 / R agrees up to sign
assert np.allclose(np.abs(matR), np.abs(matR_np)), "R disagrees with numpy"
# Q 도 / and Q too
assert np.allclose(np.abs(matQ), np.abs(matQ_np)), "Q disagrees with numpy"
print("NumPy 와 부호를 제외하고 일치 / matches NumPy up to sign")


## 더 읽을거리<br>Further reading

- 여기서는 **그람-슈미트** 로 QR 분해를 직접 만들었다. 수치적으로 더 안정한 방법으로는 **하우스홀더 반사(Householder reflection)** 와 **기븐스 회전(Givens rotation)** 이 있다.<br>
  Here we built QR by **Gram-Schmidt**. Numerically more stable alternatives are **Householder reflections** and **Givens rotations**.

- QR 분해는 일반 행렬의 *모든* 고유치를 구하는 **QR 알고리듬**(QR algorithm) 의 핵심이다. 거기서는 $A = QR$ 로 분해한 뒤 $RQ$ 를 다시 분해하기를 반복한다.<br>
  QR decomposition is the engine of the **QR algorithm** for finding *all* eigenvalues of a general matrix, where one repeatedly factors $A = QR$ and re-forms $RQ$.


## Final Bell<br>마지막 종


In [ ]:
# stackoverfow.com/a/24634221
import os
os.system("printf '\a'");
